In [4]:
def main(datasource, start_date, end_date):
    """
    多因子融合策略：结合订单簿、量价、财务、机器学习四类因子
    
    参数说明（由平台注入）:
    - datasource: 数据源字典，包含 'bar1m' 和 'financial' 两个表名
    - start_date: 评估开始日期
    - end_date: 评估结束日期
    """
    # ========== 导入依赖 ==========
    import pandas as pd
    import numpy as np
    import dai
    import lightgbm as lgb
    from datetime import timedelta
    
    # ========== 关键设置 ==========
    # 从datasource获取表名（不能硬编码！）
    bar1m_table = datasource["bar1m"]      # 分钟K线和订单簿数据表
    financial_table = datasource["financial"]  # 财务数据表
    
    # 为时序算子预留足够的历史数据窗口（往前推60天）
    lookback_days = 60
    query_start = (pd.to_datetime(start_date) - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    query_end = end_date
    
    print(f"查询数据范围: {query_start} 至 {query_end}")
    print(f"评估范围: {start_date} 至 {end_date}")
    
    # ========== 辅助函数 ==========
    
    def standardize_cross_section(df, factor_col='factor'):
        """截面标准化"""
        df[factor_col] = df.groupby('date')[factor_col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        )
        return df
    
    def winsorize(df, factor_col='factor', lower=0.01, upper=0.99):
        """去极值"""
        df[factor_col] = df.groupby('date')[factor_col].transform(
            lambda x: x.clip(x.quantile(lower), x.quantile(upper))
        )
        return df
    
    def filter_evaluation_period(df, date_col='date'):
        """只保留评估区间内的数据"""
        df = df[(df[date_col] >= start_date) & (df[date_col] <= end_date)]
        return df
    
    # ========== 因子1: 高频微观结构因子 ==========
    print("计算高频微观结构因子...")
    sql_obp = f"""
    WITH minute_data AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            -- 订单簿不平衡度
            (bid_volume1 - ask_volume1) / (bid_volume1 + ask_volume1 + 1e-8) AS obp_1,
            -- 五档加权订单簿压力
            (bid_volume1*5 + bid_volume2*4 + bid_volume3*3 + bid_volume4*2 + bid_volume5*1
             - ask_volume1*5 - ask_volume2*4 - ask_volume3*3 - ask_volume4*2 - ask_volume5*1)
            / (bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
               + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 + 1e-8) AS obp_weighted,
            -- 买卖价差
            (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2 + 1e-8) AS spread,
            -- 订单簿深度比率
            (bid_volume1 + bid_volume2 + bid_volume3) / 
                (ask_volume1 + ask_volume2 + ask_volume3 + 1e-8) AS depth_ratio
        FROM {bar1m_table}
        WHERE date >= '{query_start}' AND date <= '{query_end}'
    )
    SELECT
        date,
        instrument,
        AVG(obp_1) AS factor_obp,
        AVG(obp_weighted) AS factor_obp_w,
        AVG(spread) AS factor_spread,
        AVG(depth_ratio) AS factor_depth
    FROM minute_data
    GROUP BY date, instrument
    """
    
    try:
        df_obp = dai.query(sql_obp, compression=True).df()
        
        # 合成微观结构因子
        df_obp['factor_microstructure'] = (
            df_obp['factor_obp'] * 0.35 + 
            df_obp['factor_obp_w'] * 0.35 - 
            df_obp['factor_spread'] * 0.15 + 
            df_obp['factor_depth'] * 0.15
        )
        df_obp = standardize_cross_section(df_obp[['date', 'instrument', 'factor_microstructure']])
        
        # 只保留评估区间
        df_obp = filter_evaluation_period(df_obp)
        print(f"  微观结构因子: {len(df_obp)} 条数据")
    except Exception as e:
        print(f"  微观结构因子计算失败: {e}")
        df_obp = pd.DataFrame(columns=['date', 'instrument', 'factor_microstructure'])
    
    # ========== 因子2: 技术量价因子 ==========
    print("计算技术量价因子...")
    sql_tech = f"""
    WITH daily_data AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            LAST(close) AS close,
            FIRST(open) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            SUM(volume) AS volume,
            SUM(amount) AS amount
        FROM {bar1m_table}
        WHERE date >= '{query_start}' AND date <= '{query_end}'
        GROUP BY date::DATE, instrument
    ),
    features AS (
        SELECT
            date,
            instrument,
            close,
            -- 多周期动量
            close / LAG(close, 5) OVER (PARTITION BY instrument ORDER BY date) - 1 AS mom_5,
            close / LAG(close, 10) OVER (PARTITION BY instrument ORDER BY date) - 1 AS mom_10,
            close / LAG(close, 20) OVER (PARTITION BY instrument ORDER BY date) - 1 AS mom_20,
            -- 波动率
            STDDEV_SAMP(close / LAG(close, 1) OVER (PARTITION BY instrument ORDER BY date) - 1)
                OVER (PARTITION BY instrument ORDER BY date ROWS BETWEEN 20 PRECEDING AND CURRENT ROW) AS vol_20,
            -- 振幅
            (high - low) / (close + 1e-8) AS amplitude,
            -- 换手率
            volume / (amount + 1e-8) AS turnover
        FROM daily_data
    )
    SELECT
        date,
        instrument,
        -- 反转因子: 短期动量取负 (反转效应)
        -mom_5 AS factor_reversal,
        -- 中期动量因子
        mom_20 AS factor_momentum,
        -- 低波动率因子
        -vol_20 AS factor_lowvol,
        -- 低振幅因子
        -amplitude AS factor_lowamp,
        -- 低换手率因子
        -turnover AS factor_lowturn
    FROM features
    """
    
    try:
        df_tech = dai.query(sql_tech, compression=True).df()
        
        # 合成技术因子
        df_tech['factor_technical'] = (
            df_tech['factor_reversal'] * 0.3 + 
            df_tech['factor_momentum'] * 0.2 + 
            df_tech['factor_lowvol'] * 0.2 + 
            df_tech['factor_lowamp'] * 0.15 + 
            df_tech['factor_lowturn'] * 0.15
        )
        df_tech = standardize_cross_section(df_tech[['date', 'instrument', 'factor_technical']])
        
        # 只保留评估区间
        df_tech = filter_evaluation_period(df_tech)
        print(f"  技术量价因子: {len(df_tech)} 条数据")
    except Exception as e:
        print(f"  技术量价因子计算失败: {e}")
        df_tech = pd.DataFrame(columns=['date', 'instrument', 'factor_technical'])
    
    # ========== 因子3: 基本面因子 ==========
    print("计算基本面因子...")
    sql_fin = f"""
    SELECT
        date,
        instrument,
        net_profit_attributable_to_parent_ttm / (equity_parent_lf + 1e-8) AS roe,
        net_profit_attributable_to_parent_ttm / (market_cap + 1e-8) AS ep,
        equity_parent_lf / (market_cap + 1e-8) AS bp,
        (revenue - cost) / (revenue + 1e-8) AS gp,
        total_operating_revenue / (total_assets + 1e-8) AS at
    FROM {financial_table}
    WHERE date >= '{query_start}' AND date <= '{query_end}'
    """
    
    try:
        df_fin = dai.query(sql_fin, compression=True).df()
        
        # 合成基本面因子 (质量 + 价值)
        df_fin['factor_quality'] = (
            df_fin['roe'].rank(pct=True) * 0.4 + 
            df_fin['gp'].rank(pct=True) * 0.3 + 
            df_fin['at'].rank(pct=True) * 0.3
        )
        
        df_fin['factor_value'] = (
            -df_fin['ep'].rank(pct=True) * 0.5 +  # EP越低越好(便宜)
            -df_fin['bp'].rank(pct=True) * 0.5   # BP越低越好(便宜)
        )
        
        df_fin['factor_fundamental'] = (
            df_fin['factor_quality'] * 0.5 + 
            df_fin['factor_value'] * 0.5
        )
        
        df_fin = standardize_cross_section(df_fin[['date', 'instrument', 'factor_fundamental']])
        
        # 只保留评估区间
        df_fin = filter_evaluation_period(df_fin)
        print(f"  基本面因子: {len(df_fin)} 条数据")
    except Exception as e:
        print(f"  基本面因子计算失败: {e}")
        df_fin = pd.DataFrame(columns=['date', 'instrument', 'factor_fundamental'])
    
    # ========== 因子4: 机器学习因子 ==========
    print("计算机器学习因子...")
    
    try:
        # 准备特征数据
        sql_ml = f"""
        WITH 
        kline AS (
            SELECT
                date::DATE::DATETIME AS date,
                instrument,
                LAST(close) AS close,
                AVG(volume) AS volume,
                (bid_volume1 - ask_volume1) / (bid_volume1 + ask_volume1 + 1e-8) AS obp
            FROM {bar1m_table}
            WHERE date >= '{query_start}' AND date <= '{query_end}'
            GROUP BY date::DATE, instrument
        ),
        fin AS (
            SELECT
                date,
                instrument,
                net_profit_attributable_to_parent_ttm / (equity_parent_lf + 1e-8) AS roe
            FROM {financial_table}
            WHERE date >= '{query_start}' AND date <= '{query_end}'
        )
        SELECT
            k.date,
            k.instrument,
            k.close,
            k.volume,
            k.obp,
            f.roe,
            LEAD(k.close, 1) OVER (PARTITION BY k.instrument ORDER BY k.date) / k.close - 1 AS forward_ret
        FROM kline k
        LEFT JOIN f ON k.date = f.date AND k.instrument = f.instrument
        """
        
        df_ml = dai.query(sql_ml, compression=True).df()
        df_ml = df_ml.dropna()
        
        print(f"  ML特征数据: {len(df_ml)} 条")
        
        if len(df_ml) > 0:
            features = ['close', 'volume', 'obp', 'roe']
            dates = df_ml['date'].unique()
            split = int(len(dates) * 0.8)
            
            train_data = df_ml[df_ml['date'].isin(dates[:split])]
            
            # 训练LightGBM模型
            model = lgb.LGBMRegressor(
                n_estimators=100, max_depth=5, learning_rate=0.05,
                subsample=0.8, random_state=42, verbose=-1
            )
            model.fit(train_data[features], train_data['forward_ret'])
            df_ml['factor_ml'] = model.predict(df_ml[features])
            
            df_ml = standardize_cross_section(df_ml[['date', 'instrument', 'factor_ml']])
            
            # 只保留评估区间
            df_ml = filter_evaluation_period(df_ml.rename(columns={'factor_ml': 'factor'}))
            df_ml = df_ml.rename(columns={'factor': 'factor_ml'})
            print(f"  机器学习因子: {len(df_ml)} 条数据")
        else:
            df_ml = pd.DataFrame(columns=['date', 'instrument', 'factor_ml'])
    except Exception as e:
        print(f"  机器学习因子计算失败: {e}")
        import traceback
        traceback.print_exc()
        df_ml = pd.DataFrame(columns=['date', 'instrument', 'factor_ml'])
    
    # ========== 因子融合 ==========
    print("融合因子...")
    
    # 收集有效因子
    factor_list = []
    weight_list = []
    
    if len(df_obp) > 0:
        factor_list.append(('factor_microstructure', df_obp))
        weight_list.append(0.30)
    
    if len(df_tech) > 0:
        factor_list.append(('factor_technical', df_tech))
        weight_list.append(0.30)
    
    if len(df_fin) > 0:
        factor_list.append(('factor_fundamental', df_fin))
        weight_list.append(0.20)
    
    if len(df_ml) > 0:
        factor_list.append(('factor_ml', df_ml))
        weight_list.append(0.20)
    
    # 归一化权重
    total_weight = sum(weight_list)
    weight_list = [w / total_weight for w in weight_list]
    
    print(f"  融合 {len(factor_list)} 个因子")
    
    if len(factor_list) > 0:
        # 合并所有因子
        df_all = factor_list[0][1].rename(columns={factor_list[0][0]: 'factor'})
        
        for i in range(1, len(factor_list)):
            df_all = df_all.merge(
                factor_list[i][1].rename(columns={factor_list[i][0]: 'factor'}), 
                on=['date', 'instrument'], 
                how='outer'
            )
        
        # 重命名列
        col_names = ['factor'] + [f'factor_{i}' for i in range(1, len(factor_list))]
        df_all.columns = ['date', 'instrument'] + col_names
        
        # 加权融合
        df_all['factor'] = sum(
            df_all[f'factor_{i}'] * weight_list[i] 
            for i in range(len(factor_list))
        )
    else:
        # 如果所有因子都失败，创建空结果
        df_all = pd.DataFrame(columns=['date', 'instrument', 'factor'])
    
    # ========== 后处理 ==========
    if len(df_all) > 0:
        df_all = standardize_cross_section(df_all)
        df_all = winsorize(df_all)
        df_all = standardize_cross_section(df_all)
        
        result = df_all[['date', 'instrument', 'factor']].dropna()
        print(f"最终因子数量: {len(result)}")
        
        # 检查覆盖率
        coverage = result.groupby('date')['instrument'].nunique()
        print(f"日期范围: {result['date'].min()} 至 {result['date'].max()}")
        print(f"平均每日股票数: {coverage.mean():.0f}")
    else:
        result = df_all
        print("警告: 所有因子计算失败！")
    



In [5]:
def main(datasource, start_date, end_date):
    """
    主函数 - BigQuant平台会自动注入参数
    - datasource: 包含数据表名的字典
    - start_date: 评估开始日期
    - end_date: 评估结束日期
    
    返回：包含 date, instrument, factor 三列的DataFrame
    """
    import pandas as pd
    import numpy as np
    import dai
    from datetime import timedelta
    
    # ========== 1. 获取数据表名 ==========
    # 必须从datasource获取，不能硬编码！
    bar1m_table = datasource["bar1m"]
    financial_table = datasource["financial"]
    
    # ========== 2. 设置查询范围 ==========
    # 往前推60天，为时序算子提供足够的历史数据
    lookback_days = 60
    query_start = (pd.to_datetime(start_date) - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    query_end = end_date
    
    # ========== 3. 计算高频订单簿因子 ==========
    # 基于1分钟订单簿数据，计算日内订单簿压力指标
    sql_obp = f"""
    WITH 
    -- 步骤1: 计算1分钟级别的订单簿指标
    minute_features AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            -- 买一卖一压力: (买量-卖量)/(买量+卖量)
            (bid_volume1 - ask_volume1) / (bid_volume1 + ask_volume1 + 1e-8) AS obp_level1,
            -- 三档压力
            (bid_volume1 + bid_volume2 + bid_volume3 - ask_volume1 - ask_volume2 - ask_volume3)
                / (bid_volume1 + bid_volume2 + bid_volume3 + ask_volume1 + ask_volume2 + ask_volume3 + 1e-8) AS obp_level3,
            -- 五档加权压力（深度加权，越靠近中间权重越大）
            (bid_volume1*5 + bid_volume2*4 + bid_volume3*3 + bid_volume4*2 + bid_volume5*1
             - ask_volume1*5 - ask_volume2*4 - ask_volume3*3 - ask_volume4*2 - ask_volume5*1)
            / (bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
               + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 + 1e-8) AS obp_weighted,
            -- 买卖价差 (相对值)
            (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2 + 1e-8) AS spread,
            -- 订单簿深度 (总买量/总卖量)
            (bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5) / 
                (ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 + 1e-8) AS depth,
            -- 价格
            LAST(close) AS close
        FROM {bar1m_table}
        WHERE date >= '{query_start}' AND date <= '{query_end}'
    ),
    -- 步骤2: 聚合为日频因子
    daily_features AS (
        SELECT
            date,
            instrument,
            -- 日内平均订单簿压力
            AVG(obp_level1) AS obp_mean,
            -- 三档平均压力
            AVG(obp_level3) AS obp_3level_mean,
            -- 加权平均压力
            AVG(obp_weighted) AS obp_weighted_mean,
            -- 平均价差
            AVG(spread) AS avg_spread,
            -- 平均深度
            AVG(depth) AS avg_depth,
            -- 当日收盘价
            LAST(close) AS close
        FROM minute_features
        GROUP BY date, instrument
    ),
    -- 步骤3: 计算技术指标
    with_technicals AS (
        SELECT
            date,
            instrument,
            obp_mean,
            obp_3level_mean,
            obp_weighted_mean,
            avg_spread,
            avg_depth,
            close,
            -- 5日动量
            close / LAG(close, 5) OVER (PARTITION BY instrument ORDER BY date) - 1 AS mom_5,
            -- 20日动量  
            close / LAG(close, 20) OVER (PARTITION BY instrument ORDER BY date) - 1 AS mom_20,
            -- 20日波动率
            STDDEV_SAMP(close / LAG(close, 1) OVER (PARTITION BY instrument ORDER BY date) - 1)
                OVER (PARTITION BY instrument ORDER BY date ROWS BETWEEN 20 PRECEDING AND CURRENT ROW) AS vol_20
        FROM daily_features
    )
    SELECT
        date,
        instrument,
        obp_mean,
        obp_3level_mean,
        obp_weighted_mean,
        avg_spread,
        avg_depth,
        mom_5,
        mom_20,
        vol_20
    FROM with_technicals
    """
    
    print("正在查询高频数据...")
    df = dai.query(sql_obp, compression=True).df()
    print(f"获取到 {len(df)} 条数据")
    
    # ========== 4. 因子合成 ==========
    # 方法: 多因子加权合成
    # 
    # 因子逻辑:
    # 1. 订单簿压力 (obp): 多头压力越大越好 (正因子)
    # 2. 买卖价差 (spread): 价差越小越好 (负因子)
    # 3. 深度 (depth): 深度越大越好 (正因子)  
    # 4. 短期反转 (mom_5): 前5天跌的越多越好 (负因子)
    # 5. 中期动量 (mom_20): 前20天涨的越多越好 (正因子)
    # 6. 低波动 (vol_20): 波动率越低越好 (负因子)
    
    df['factor'] = (
        df['obp_mean'] * 0.20 +           # 订单簿压力
        df['obp_weighted_mean'] * 0.15 +   # 加权订单簿压力
        -df['avg_spread'] * 0.10 +         # 低价差
        df['avg_depth'] * 0.10 +           # 高深度
        -df['mom_5'] * 0.15 +              # 短期反转
        df['mom_20'] * 0.15 +              # 中期动量
        -df['vol_20'] * 0.15               # 低波动
    )
    
    # ========== 5. 截面标准化 ==========
    # 每天对所有股票进行标准化
    df['factor'] = df.groupby('date')['factor'].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )
    
    # ========== 6. 去极值处理 ==========
    # 去掉1%和99%分位数之外的极端值
    df['factor'] = df.groupby('date')['factor'].transform(
        lambda x: x.clip(x.quantile(0.01), x.quantile(0.99))
    )
    
    # ========== 7. 最终标准化 ==========
    df['factor'] = df.groupby('date')['factor'].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )
    
    # ========== 8. 筛选评估区间 ==========
    # 只保留评估区间内的数据
    result = df[
        (df['date'] >= start_date) & 
        (df['date'] <= end_date)
    ][['date', 'instrument', 'factor']].dropna()
    
    print(f"最终因子数量: {len(result)}")
    
    # 检查覆盖率
    if len(result) > 0:
        daily_count = result.groupby('date')['instrument'].nunique()
        print(f"日期范围: {result['date'].min()} 至 {result['date'].max()}")
        print(f"平均每日覆盖股票数: {daily_count.mean():.0f}")
    
    # ========== 9. 返回因子数据表格 ==========
    # 必须返回包含 date, instrument, factor 三列的DataFrame
    print("因子计算完成，返回结果...")
    return result